# Sensitivity Analysis of Latent Dimensions for SAMVAE

This notebook analyzes how different latent dimension configurations affect model performance (CI-IBS metric) across modalities in the hyperparameter optimization results. The notebook:

1. Extracts hyperparameter optimization results from the results directory

2. Filters latent dimensions of interest:

   - **Clinical modality**: 5, 10. The boxplots display latent dimensions on the x-axis and CI-IBS performance on the y-axis, with different colors representing different modality combinations.

   - **Multimodal combinations**: 5, 50, 500 (for the omic/image dimension)

3. Selects the **top N best performing configurations** (N=3 by default) based on CI-IBS metric. Generates boxplots showing the distribution of these top N values across different latent dimensions

## 1. Configuration

In [13]:
import os
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Tuple, List

# Configuration
SCRIPT_CONFIG = {
    "results_dir": os.path.join("..", "results", "Hyperparameter_optimization"),
    "output_dir": os.path.join("figs", "sensitive_analysis_samvae"),
    "datasets": ["brca", "lgg"],
    "analyses": ["Survival_Analysis", "Competing_Risks"],
    "top_n": 3
}

## 2. Helper Functions

In [14]:
def find_result_table_path(base_dir: str, analysis_type: str) -> str:
    if analysis_type == "Competing_Risks":
        candidates = [os.path.join(base_dir, "total_results_table_cr.tex")]
    else:
        candidates = [os.path.join(base_dir, "total_results_table.tex")]
    for p in candidates:
        if os.path.isfile(p):
            return p
    return ""


def parse_tex_table(path: str) -> Tuple[List[str], List[List[str]]]:
    headers: List[str] = []
    rows: List[List[str]] = []
    if not path:
        return headers, rows
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("%") or "&" not in line:
                continue
            if line.endswith("\\\\"):
                line = line[:-2].strip()
            parts = [p.strip() for p in line.split("&")]
            if ("DATASETS" in parts and "LATENT" in parts) or ("LATENT" in parts and "HIDDEN" in parts):
                headers = parts
                continue
            if headers and len(parts) >= len(headers):
                rows.append(parts[:len(headers)])
    return headers, rows


_MEAN_STD_RE = re.compile(r"([0-9]*\.?[0-9]+)\s*±\s*([0-9]*\.?[0-9]+)")


def parse_value(v: str) -> float:
    v = v.strip()
    m = _MEAN_STD_RE.match(v)
    if m:
        return float(m.group(1))
    try:
        return float(v)
    except Exception:
        return float("nan")

## 3. Data Collection

In [15]:
# Collect all records from hyperparameter optimization results:
def collect_records(results_root: str) -> pd.DataFrame:
    records = []

    def pick_table(mode_dir: str, analysis: str) -> str:
        dirs = [mode_dir]
        for sub in os.listdir(mode_dir):
            p = os.path.join(mode_dir, sub)
            if os.path.isdir(p):
                dirs.append(p)
        for d in dirs:
            table_path = find_result_table_path(d, analysis)
            if table_path:
                return table_path
        return ""

    def parse_latents(entry: str) -> List[int]:
        cleaned = entry.replace("[", "").replace("]", "")
        parts = [p.strip() for p in cleaned.split(",") if p.strip()]
        return [int(float(p)) for p in parts]

    for analysis in ["Competing_Risks", "Survival_Analysis"]:
        at_dir = os.path.join(results_root, analysis)
        if not os.path.isdir(at_dir):
            continue
        for dataset in sorted(os.listdir(at_dir)):
            ds_dir = os.path.join(at_dir, dataset)
            if not os.path.isdir(ds_dir):
                continue
            for mode in sorted(os.listdir(ds_dir)):
                mode_dir = os.path.join(ds_dir, mode)
                if not os.path.isdir(mode_dir):
                    continue
                table_path = pick_table(mode_dir, analysis)
                if not table_path:
                    continue
                headers, rows = parse_tex_table(table_path)
                if not headers or not rows:
                    continue
                hmap = {h: i for i, h in enumerate(headers)}
                if "LATENT" not in hmap or "C-INDEX" not in hmap or "IBS" not in hmap:
                    if "CI - IBS" not in hmap:
                        continue
                for r in rows:
                    try:
                        latents = parse_latents(r[hmap["LATENT"]])
                        risk = int(float(r[hmap["RISK"]])) if "RISK" in hmap else 0
                        if "C-INDEX" in hmap and "IBS" in hmap:
                            val = parse_value(r[hmap["C-INDEX"]]) - parse_value(r[hmap["IBS"]])
                        else:
                            val = parse_value(r[hmap["CI - IBS"]])
                        target_latent = latents[0] if mode == "clinical" else (latents[1] if len(latents) > 1 else latents[0])
                        records.append({
                            "analysis": analysis,
                            "dataset": dataset,
                            "mode": mode,
                            "risk": risk,
                            "target_latent": target_latent,
                            "label": f"{mode}_{target_latent}",
                            "ci_minus_ibs": val,
                        })
                    except Exception:
                        continue

    return pd.DataFrame(records)

## 4. Load Data

In [16]:
# Load all hyperparameter optimization results:
results_dir = os.path.abspath(SCRIPT_CONFIG["results_dir"])
df = collect_records(results_dir)

if df.empty:
    print("No records found. Please verify results_dir.")
else:
    display(df.head(10))

,analysis,dataset,mode,risk,target_latent,label,ci_minus_ibs
0,Competing_Risks,brca,clinical,0,5,clinical_5,0.426
1,Competing_Risks,brca,clinical,1,5,clinical_5,0.390
2,Competing_Risks,brca,clinical,0,5,clinical_5,0.338
3,Competing_Risks,brca,clinical,1,5,clinical_5,0.457
4,Competing_Risks,brca,clinical,0,5,clinical_5,0.391
5,Competing_Risks,brca,clinical,1,5,clinical_5,0.425
6,Competing_Risks,brca,clinical,0,5,clinical_5,0.387
7,Competing_Risks,brca,clinical,1,5,clinical_5,0.385
8,Competing_Risks,brca,clinical,0,5,clinical_5,0.379
9,Competing_Risks,brca,clinical,1,5,clinical_5,0.403


## 5. Plotting Functions

In [17]:
# Function to generate a boxplot showing all modality-latent combinations
def plot_boxplots_all_combinations(df, output_dir, datasets=None, analyses=None, top_n=3):
    if df.empty:
        return pd.DataFrame()
    if datasets:
        df = df[df["dataset"].isin(datasets)]
    if analyses:
        df = df[df["analysis"].isin(analyses)]
    is_clinical = df["mode"] == "clinical"
    keep_clinical = is_clinical & df["target_latent"].isin([5, 10])
    keep_omic = (~is_clinical) & df["target_latent"].isin([5, 50, 500])
    df_filtered = df[keep_clinical | keep_omic].copy()
    df_top = (
        df_filtered.sort_values("ci_minus_ibs", ascending=False)
        .groupby(["analysis", "dataset", "risk", "label"])
        .head(top_n)
    )
    sns.set_style("whitegrid")
    unique_groups = df_top.groupby(["analysis", "dataset", "risk"])
    for (analysis, dataset, risk), group_data in unique_groups:
        group_data = group_data.copy()
        group_data["sort_key"] = group_data.apply(
            lambda row: (row["mode"], row["target_latent"]), axis=1
        )
        group_data = group_data.sort_values("sort_key")

        def format_modality_name(mode):
            if mode == "clinical":
                return "Clinical"
            if "clinical_omic_" in mode:
                omic_type = mode.replace("clinical_omic_", "")
                if omic_type == "RNAseq":
                    return "Clinical + RNAseq"
                if omic_type == "adn":
                    return "Clinical + DNA"
                if omic_type == "cnv":
                    return "Clinical + CNV"
                if omic_type == "miRNA":
                    return "Clinical + miRNA"
                return f"Clinical + {omic_type}"
            if "patch" in mode.lower():
                return "Clinical + 1 patch"
            return mode
        group_data["display_mode"] = group_data["mode"].apply(format_modality_name)
        fig, ax = plt.subplots(figsize=(max(10, 0.5 * group_data["label"].nunique()), 6))
        analysis_abbr = "CR" if analysis == "Competing_Risks" else "SA"
        if analysis == "Competing_Risks":
            title = f"{dataset.upper()}-{analysis_abbr} - Sensitivity Analysis of Latent Dimensions (Risk {risk})"
        else:
            title = f"{dataset.upper()}-{analysis_abbr} - Sensitivity Analysis of Latent Dimensions"
        ax.set_title(title, fontsize=16)
        sns.boxplot(
            data=group_data,
            x="label",
            y="ci_minus_ibs",
            hue="display_mode",
            dodge=False,
            palette="tab10",
            order=group_data["label"].unique(),
            ax=ax,
        )
        tick_positions = range(len(group_data["label"].unique()))
        ax.set_xticks(tick_positions)
        ax.set_xticklabels([
            str(group_data[group_data["label"] == lbl]["target_latent"].iloc[0])
            for lbl in group_data["label"].unique()
        ])
        ax.set_xlabel("Latent Dimension of Modalities", fontsize=14)
        ax.set_ylabel("CI - IBS", fontsize=14)
        ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, fontsize=15)
        plt.tight_layout()
        out_sub = os.path.join(output_dir, analysis, dataset)
        os.makedirs(out_sub, exist_ok=True)
        fname = f"boxplot{'_risk'+str(risk) if analysis == 'Competing_Risks' else ''}.png"
        plt.savefig(os.path.join(out_sub, fname), dpi=300, bbox_inches="tight")
        plt.close()
    summary = (
        df_top.groupby(["analysis", "dataset", "risk", "label"])["ci_minus_ibs"]
        .agg(Top_Values=lambda x: list(np.round(x, 4)), Mean="mean")
        .reset_index()
    )
    return summary

## 6. Generate Plots

In [18]:
output_dir = os.path.abspath(SCRIPT_CONFIG["output_dir"])
summary = plot_boxplots_all_combinations(
    df,
    output_dir,
    datasets=SCRIPT_CONFIG["datasets"],
    analyses=SCRIPT_CONFIG["analyses"],
    top_n=SCRIPT_CONFIG["top_n"],
 )
if not summary.empty:
    display(summary)

,analysis,dataset,risk,label,Top_Values,Mean
0,Competing_Risks,brca,0,clinical_10,"[0.424, 0.403, 0.399]",0.408667
1,Competing_Risks,brca,0,clinical_5,"[0.426, 0.395, 0.391]",0.404000
2,Competing_Risks,brca,0,clinical_omic_RNAseq_5,"[0.404, 0.392, 0.374]",0.390000
3,Competing_Risks,brca,0,clinical_omic_RNAseq_50,"[0.413, 0.412, 0.365]",0.396667
4,Competing_Risks,brca,0,clinical_omic_RNAseq_500,"[0.425, 0.388, 0.379]",0.397333
...,...,...,...,...,...,...
97,Survival_Analysis,lgg,0,clinical_omic_miRNA_50,"[0.451, 0.43, 0.385]",0.422000
98,Survival_Analysis,lgg,0,clinical_omic_miRNA_500,"[0.388, 0.385, 0.362]",0.378333
99,Survival_Analysis,lgg,0,clinical_wsi_patches_1_patch_5,"[0.518, 0.485, 0.454]",0.485667
100,Survival_Analysis,lgg,0,clinical_wsi_patches_1_patch_50,"[0.458, 0.418, 0.404]",0.426667
